# Notebook 06: Statistical Validation and Error Analysis

This notebook is validation-only. It does not train models, load checkpoints, tune thresholds, or select models.

It implements the thesis Section 3.7 validation layer:

- Table 3.40 pairwise comparisons
- Table 3.44 McNemar statistical significance table
- Section 3.7.6 confusion/error analysis
- Section 3.7.4 GA weight stability and interpretability checks

Notebook 05 created reporting tables. Notebook 06 validates model differences using paired prediction rows.

## Setup

In [ ]:
from pathlib import Path
import os, re, json, math, shutil, zipfile, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import chi2, binomtest, spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

MODEL_ORDER = [
    "tfidf_baseline1",
    "tfidf_ablation_a",
    "distilbert_baseline2",
    "distilbert_ablation_b",
    "ablation_c",
    "ablation_d",
    "proposed_ga_v3",
]
DISPLAY_NAMES = {
    "tfidf_baseline1": "Baseline 1 (TF-IDF, unweighted)",
    "tfidf_ablation_a": "Ablation A (TF-IDF, class-weighted)",
    "distilbert_baseline2": "Baseline 2 (DistilBERT fine-tuned)",
    "distilbert_ablation_b": "Ablation B (DistilBERT + augmentation)",
    "ablation_c": "Ablation C (Frozen DistilBERT + uniform)",
    "ablation_d": "Ablation D (Frozen DistilBERT + random)",
    "proposed_ga_v3": "Proposed Model (GA-optimized fusion)",
}
SPLITS = ["test_clean", "test_adv_10", "test_adv_20", "test_adv_30"]
SPLIT_LABELS = {"test_clean": "clean", "test_adv_10": "adv10", "test_adv_20": "adv20", "test_adv_30": "adv30"}
SEED_PRIORITY = [42, 7, 123]
ALPHA = 0.05
ALPHA_PER_SPLIT_BONFERRONI = ALPHA / 9
ALPHA_GLOBAL_BONFERRONI = ALPHA / 36
PAIRWISE_COMPARISONS = [
    ("proposed_ga_v3", "tfidf_baseline1", "Proposed vs. Baseline 1", "GA fusion vs. traditional ML"),
    ("proposed_ga_v3", "distilbert_baseline2", "Proposed vs. Baseline 2", "GA fusion (frozen) vs. fine-tuned DistilBERT"),
    ("proposed_ga_v3", "tfidf_ablation_a", "Proposed vs. Ablation A", "GA fusion vs. cost-sensitive TF-IDF"),
    ("proposed_ga_v3", "distilbert_ablation_b", "Proposed vs. Ablation B", "GA weights + features vs. augmentation alone"),
    ("proposed_ga_v3", "ablation_c", "Proposed vs. Ablation C", "GA-optimized vs. uniform weights"),
    ("proposed_ga_v3", "ablation_d", "Proposed vs. Ablation D", "GA-optimized vs. random weights"),
    ("ablation_c", "ablation_d", "Ablation C vs. Ablation D", "Uniform vs. random weights"),
    ("tfidf_baseline1", "tfidf_ablation_a", "Baseline 1 vs. Ablation A", "Unweighted vs. class-weighted TF-IDF"),
    ("distilbert_baseline2", "distilbert_ablation_b", "Baseline 2 vs. Ablation B", "Standard vs. augmented DistilBERT"),
]
G_FEATURES = [
    "G1_URL_Signals", "G2_OTP_Numeric_Density", "G3_Obfuscation", "G4_Urgency_Threat_Cues",
    "G5_Action_Directives", "G6_Financial_Terms", "G7_Auth_Secrets_Request", "G8_Brand_Impersonation",
]
print("Setup complete. SciPy available:", SCIPY_AVAILABLE)

## Detect FinalAllTrained Root and Output Folders

In [ ]:
CONTENT_DIR = Path("/content") if (os.name != "nt" and Path("/content").exists()) else Path.cwd()
candidates = [CONTENT_DIR / "FinalAllTrained", CONTENT_DIR / "thesis-modeling" / "FinalAllTrained", Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = None
for c in candidates:
    if (c / "results" / "predictions").exists() and (c / "results" / "metrics").exists() and (c / "artifacts").exists():
        PROJECT_ROOT = c.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate FinalAllTrained root with results/predictions, results/metrics, and artifacts.")

PRED_DIR = PROJECT_ROOT / "results" / "predictions"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
GA_DIR = PROJECT_ROOT / "artifacts" / "ga_runs" / "proposed_ga_v3"
OUT_DIR = PROJECT_ROOT / "results" / "statistical_validation"
FIG_DIR = PROJECT_ROOT / "results" / "figures" / "statistical_validation"
REPORT_DIR = PROJECT_ROOT / "reports" / "statistical_validation"
for d in [OUT_DIR, FIG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Output folder =", OUT_DIR)

## Representative Seed Selection

In [ ]:
def metric_file_for_model(model_key):
    folder = METRICS_DIR / model_key
    candidates = [
        folder / f"{model_key}_metrics_by_seed.csv",
        folder / f"{model_key}_metrics.csv",
        folder / "ablation_c_metrics.csv",
        folder / "proposed_ga_v3_metrics_by_seed.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def median_seed_from_metrics(model_key, split):
    p = metric_file_for_model(model_key)
    if p is None:
        return None, "missing metrics file"
    df = pd.read_csv(p)
    if "seed" not in df.columns:
        return 42, "no seed column; defaulted to 42"
    sub = df[df["split"].eq(split)].copy()
    if sub.empty:
        return 42, f"no metrics row for {split}; defaulted to 42"
    if sub["seed"].nunique() == 1:
        return int(sub["seed"].iloc[0]), "single available seed"
    sub["seed_priority"] = sub["seed"].map({s:i for i,s in enumerate(SEED_PRIORITY)}).fillna(99)
    unique_f1 = sorted(sub["f1_smishing"].dropna().unique())
    if len(unique_f1) == 1:
        selected = int(sub.sort_values("seed_priority").iloc[0]["seed"])
        return selected, "all seeds tied on F1; tie priority 42, then 7, then 123"
    median_f1 = sub["f1_smishing"].median()
    sub["distance_to_median_f1"] = (sub["f1_smishing"] - median_f1).abs()
    sub = sub.sort_values(["distance_to_median_f1", "seed_priority"], ascending=[True, True]).reset_index(drop=True)
    selected = int(sub.iloc[0]["seed"])
    return selected, "seed nearest median F1 on same split; tie priority 42, then 7, then 123"

seed_rows = []
for model_key in MODEL_ORDER:
    for split in SPLITS:
        seed, rule = median_seed_from_metrics(model_key, split)
        seed_rows.append({"model_key": model_key, "display_name": DISPLAY_NAMES[model_key], "split": split, "selected_seed": seed, "selection_rule": rule})
representative_seed_df = pd.DataFrame(seed_rows)
representative_seed_df.to_csv(OUT_DIR / "representative_seed_selection.csv", index=False)
display(representative_seed_df)

## Load Representative Predictions and Pairing Audit

In [ ]:
def prediction_file_for(model_key, split, seed):
    folder = PRED_DIR / model_key
    patterns = []
    if seed is not None:
        patterns += [f"*seed{seed}_predictions_{split}.csv"]
    patterns += [f"{model_key}_predictions_{split}.csv", f"*predictions_{split}.csv"]
    matches = []
    for pat in patterns:
        matches = sorted(folder.glob(pat))
        if matches:
            return matches[0]
    return None


def load_prediction(model_key, split, seed):
    p = prediction_file_for(model_key, split, seed)
    if p is None:
        raise FileNotFoundError(f"Missing prediction file for {model_key}, {split}, seed {seed}")
    df = pd.read_csv(p)
    required = {"final_row_id", "true_label", "predicted_label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{p} missing required columns: {missing}")
    out = df.copy()
    out["final_row_id"] = out["final_row_id"].astype(str)
    out["true_label"] = out["true_label"].astype(int)
    out["predicted_label"] = out["predicted_label"].astype(int)
    out["correct"] = out["true_label"].eq(out["predicted_label"])
    return out, p

predictions = {}
audit_rows = []
for _, row in representative_seed_df.iterrows():
    model_key, split, seed = row["model_key"], row["split"], int(row["selected_seed"])
    df, path = load_prediction(model_key, split, seed)
    predictions[(model_key, split)] = df
    audit_rows.append({
        "model_key": model_key,
        "split": split,
        "selected_seed": seed,
        "source_file": str(path.relative_to(PROJECT_ROOT)),
        "n_rows": len(df),
        "final_row_id_unique": bool(df["final_row_id"].is_unique),
        "support_ham": int((df["true_label"] == 0).sum()),
        "support_smishing": int((df["true_label"] == 1).sum()),
    })

# Pairing audit by split across all seven representative files.
for split in SPLITS:
    base_ids = None
    base_labels = None
    for model_key in MODEL_ORDER:
        df = predictions[(model_key, split)].sort_values("final_row_id")
        ids = df["final_row_id"].tolist()
        labels = df["true_label"].tolist()
        if base_ids is None:
            base_ids, base_labels = ids, labels
        else:
            if ids != base_ids:
                raise ValueError(f"final_row_id mismatch for {model_key} on {split}")
            if labels != base_labels:
                raise ValueError(f"true_label mismatch for {model_key} on {split}")

prediction_pairing_audit = pd.DataFrame(audit_rows)
prediction_pairing_audit.to_csv(OUT_DIR / "prediction_pairing_audit.csv", index=False)
print("Prediction pairing audit passed for all representative runs.")
display(prediction_pairing_audit.head())

## McNemar Pairwise Statistical Tests

In [ ]:
def mcnemar_test_from_counts(b, c):
    discordant = b + c
    if discordant == 0:
        return {"statistic": 0.0, "p_value": 1.0, "method": "no discordant pairs"}
    if discordant < 25 or not SCIPY_AVAILABLE:
        # Exact two-sided binomial test under p=0.5. Fallback uses direct sum.
        k = min(b, c)
        if SCIPY_AVAILABLE:
            p = float(binomtest(k, discordant, 0.5, alternative="two-sided").pvalue)
        else:
            tail = sum(math.comb(discordant, i) * (0.5 ** discordant) for i in range(0, k + 1))
            p = min(1.0, 2.0 * tail)
        return {"statistic": float(k), "p_value": p, "method": "exact binomial McNemar"}
    stat = ((abs(b - c) - 1) ** 2) / discordant
    if SCIPY_AVAILABLE:
        p = float(chi2.sf(stat, 1))
    else:
        # Wilson-Hilferty-ish fallback is not used for final if scipy exists; conservative placeholder.
        p = float(math.exp(-0.5 * stat))
    return {"statistic": float(stat), "p_value": p, "method": "chi-square McNemar with continuity correction"}

mcnemar_rows = []
contingency_rows = []
for split in SPLITS:
    for model_a, model_b, comparison, interpretation in PAIRWISE_COMPARISONS:
        a = predictions[(model_a, split)].sort_values("final_row_id").reset_index(drop=True)
        bdf = predictions[(model_b, split)].sort_values("final_row_id").reset_index(drop=True)
        a_correct = a["correct"].to_numpy()
        b_correct = bdf["correct"].to_numpy()
        both_correct = int(np.sum(a_correct & b_correct))
        a_correct_b_wrong = int(np.sum(a_correct & ~b_correct))
        a_wrong_b_correct = int(np.sum(~a_correct & b_correct))
        both_wrong = int(np.sum(~a_correct & ~b_correct))
        test = mcnemar_test_from_counts(a_correct_b_wrong, a_wrong_b_correct)
        row = {
            "split": split,
            "split_label": SPLIT_LABELS[split],
            "comparison": comparison,
            "interpretation": interpretation,
            "model_a": model_a,
            "model_b": model_b,
            "model_a_display": DISPLAY_NAMES[model_a],
            "model_b_display": DISPLAY_NAMES[model_b],
            "model_a_seed": int(representative_seed_df.query("model_key == @model_a and split == @split")["selected_seed"].iloc[0]),
            "model_b_seed": int(representative_seed_df.query("model_key == @model_b and split == @split")["selected_seed"].iloc[0]),
            "both_correct": both_correct,
            "model_a_correct_model_b_wrong": a_correct_b_wrong,
            "model_a_wrong_model_b_correct": a_wrong_b_correct,
            "both_wrong": both_wrong,
            "discordant_pairs": a_correct_b_wrong + a_wrong_b_correct,
            **test,
        }
        row["significant_uncorrected_0_05"] = bool(row["p_value"] < ALPHA)
        row["significant_bonferroni_per_split"] = bool(row["p_value"] < ALPHA_PER_SPLIT_BONFERRONI)
        row["significant_bonferroni_global"] = bool(row["p_value"] < ALPHA_GLOBAL_BONFERRONI)
        row["alpha_per_split_bonferroni"] = ALPHA_PER_SPLIT_BONFERRONI
        row["alpha_global_bonferroni"] = ALPHA_GLOBAL_BONFERRONI
        mcnemar_rows.append(row)
        contingency_rows.append({k: row[k] for k in ["split", "comparison", "both_correct", "model_a_correct_model_b_wrong", "model_a_wrong_model_b_correct", "both_wrong"]})

mcnemar_df = pd.DataFrame(mcnemar_rows)
contingency_df = pd.DataFrame(contingency_rows)
mcnemar_df.to_csv(OUT_DIR / "mcnemar_pairwise_results.csv", index=False)
contingency_df.to_csv(OUT_DIR / "mcnemar_contingency_tables.csv", index=False)
summary = mcnemar_df.groupby("split_label").agg(
    comparisons=("comparison", "count"),
    significant_uncorrected=("significant_uncorrected_0_05", "sum"),
    significant_bonferroni_per_split=("significant_bonferroni_per_split", "sum"),
    significant_bonferroni_global=("significant_bonferroni_global", "sum"),
).reset_index()
summary.to_csv(OUT_DIR / "mcnemar_bonferroni_summary.csv", index=False)
display(mcnemar_df.head(12))

## Confusion Matrix and Error Analysis

In [ ]:
conf_rows = []
fn_rows = []
fp_rows = []
for model_key in MODEL_ORDER:
    for split in SPLITS:
        df = predictions[(model_key, split)]
        y = df["true_label"].to_numpy()
        pred = df["predicted_label"].to_numpy()
        tp = int(np.sum((y == 1) & (pred == 1)))
        tn = int(np.sum((y == 0) & (pred == 0)))
        fp = int(np.sum((y == 0) & (pred == 1)))
        fn = int(np.sum((y == 1) & (pred == 0)))
        conf_rows.append({
            "model_key": model_key, "display_name": DISPLAY_NAMES[model_key], "split": split, "split_label": SPLIT_LABELS[split],
            "tp": tp, "tn": tn, "fp": fp, "fn": fn, "support_ham": int((y==0).sum()), "support_smishing": int((y==1).sum()),
            "fnr": fn / (fn + tp) if (fn + tp) else 0.0,
            "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        })
        fn_subset = df[(df["true_label"] == 1) & (df["predicted_label"] == 0)].copy()
        fp_subset = df[(df["true_label"] == 0) & (df["predicted_label"] == 1)].copy()
        fn_rows.append({"model_key": model_key, "display_name": DISPLAY_NAMES[model_key], "split": split, "false_negative_count": len(fn_subset)})
        fp_rows.append({"model_key": model_key, "display_name": DISPLAY_NAMES[model_key], "split": split, "false_positive_count": len(fp_subset)})

confusion_df = pd.DataFrame(conf_rows)
fn_breakdown = pd.DataFrame(fn_rows)
fp_breakdown = pd.DataFrame(fp_rows)
confusion_df.to_csv(OUT_DIR / "confusion_matrix_representative_runs.csv", index=False)
fn_breakdown.to_csv(OUT_DIR / "false_negative_breakdown_by_model_split.csv", index=False)
fp_breakdown.to_csv(OUT_DIR / "false_positive_breakdown_by_model_split.csv", index=False)

# Adversarial FN shift: clean TP that becomes FN under adversarial split.
shift_rows = []
for model_key in MODEL_ORDER:
    clean = predictions[(model_key, "test_clean")][["final_row_id", "true_label", "predicted_label"]].copy()
    clean["clean_tp"] = clean["true_label"].eq(1) & clean["predicted_label"].eq(1)
    for adv_split in ["test_adv_10", "test_adv_20", "test_adv_30"]:
        adv = predictions[(model_key, adv_split)][["final_row_id", "true_label", "predicted_label"]].copy()
        adv["adv_fn"] = adv["true_label"].eq(1) & adv["predicted_label"].eq(0)
        merged = clean.merge(adv[["final_row_id", "adv_fn"]], on="final_row_id", how="inner")
        shifted = int(np.sum(merged["clean_tp"] & merged["adv_fn"]))
        clean_tp = int(np.sum(merged["clean_tp"]))
        shift_rows.append({
            "model_key": model_key, "display_name": DISPLAY_NAMES[model_key], "adversarial_split": adv_split,
            "clean_tp_to_adversarial_fn_count": shifted,
            "clean_tp_count": clean_tp,
            "shift_rate_among_clean_tp": shifted / clean_tp if clean_tp else 0.0,
        })
adversarial_fn_shift = pd.DataFrame(shift_rows)
adversarial_fn_shift.to_csv(OUT_DIR / "adversarial_fn_shift_summary.csv", index=False)
display(confusion_df.head(12))

## GA Interpretability and Weight Stability Validation

In [ ]:
weight_files = []
for seed in SEED_PRIORITY:
    p = GA_DIR / f"ga_seed{seed}_best_weights_v3.csv"
    if p.exists():
        df = pd.read_csv(p)
        df["seed"] = seed
        weight_files.append(df)
if weight_files:
    weights_long = pd.concat(weight_files, ignore_index=True)
    weights_wide = weights_long.pivot(index="seed", columns="feature", values="weight").reindex(SEED_PRIORITY)
else:
    weights_long = pd.DataFrame()
    weights_wide = pd.DataFrame()

corr_rows = []
if not weights_wide.empty:
    for i, seed_a in enumerate(SEED_PRIORITY):
        for seed_b in SEED_PRIORITY[i+1:]:
            if seed_a in weights_wide.index and seed_b in weights_wide.index:
                a = weights_wide.loc[seed_a, G_FEATURES].to_numpy(dtype=float)
                b = weights_wide.loc[seed_b, G_FEATURES].to_numpy(dtype=float)
                if SCIPY_AVAILABLE:
                    rho, pval = spearmanr(a, b)
                    rho, pval = float(rho), float(pval)
                else:
                    rho = float(pd.Series(a).rank().corr(pd.Series(b).rank(), method="pearson"))
                    pval = np.nan
                corr_rows.append({"seed_a": seed_a, "seed_b": seed_b, "spearman_rho": rho, "p_value": pval, "meets_target_rho_ge_0_80": bool(rho >= 0.80)})

ga_corr = pd.DataFrame(corr_rows)
ga_corr.to_csv(OUT_DIR / "ga_weight_spearman_rank_correlations.csv", index=False)

if not weights_long.empty:
    top_rows = []
    for seed, grp in weights_long.groupby("seed"):
        top = grp.sort_values("weight", ascending=False).head(3)
        top_rows.append({"seed": seed, "top3_features": ", ".join(top["feature"].tolist()), "top1_feature": top.iloc[0]["feature"]})
    top_stability = pd.DataFrame(top_rows)
else:
    top_stability = pd.DataFrame(columns=["seed", "top3_features", "top1_feature"])
top_stability.to_csv(OUT_DIR / "ga_weight_top_group_stability.csv", index=False)

selected_path = GA_DIR / "ga_selected_final_weights_v3.csv"
selected_weights = pd.read_csv(selected_path) if selected_path.exists() else pd.DataFrame()
selected_threshold_path = GA_DIR / "selected_threshold_v3.json"
selected_threshold = json.loads(selected_threshold_path.read_text(encoding="utf-8")) if selected_threshold_path.exists() else {}

manifest = {
    "notebook": "06_statistical_validation_and_error_analysis.ipynb",
    "source_root": str(PROJECT_ROOT),
    "splits": SPLITS,
    "models": MODEL_ORDER,
    "pairing_key": "final_row_id",
    "mcnemar_comparisons": [c[2] for c in PAIRWISE_COMPARISONS],
    "alpha": ALPHA,
    "alpha_per_split_bonferroni": ALPHA_PER_SPLIT_BONFERRONI,
    "alpha_global_bonferroni": ALPHA_GLOBAL_BONFERRONI,
    "representative_seed_rule": "median F1 on same split; tie priority 42, 7, 123",
    "scipy_available": SCIPY_AVAILABLE,
}
(OUT_DIR / "statistical_validation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("GA interpretability validation tables saved.")
display(ga_corr)
display(top_stability)

## Validation Figures

In [ ]:
# McNemar significance heatmap: -log10 p-values by comparison and split.
heat = mcnemar_df.pivot(index="comparison", columns="split_label", values="p_value").reindex([c[2] for c in PAIRWISE_COMPARISONS])
heat_val = -np.log10(heat.astype(float).clip(lower=1e-300))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heat_val.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(heat_val.columns))); ax.set_xticklabels(heat_val.columns)
ax.set_yticks(range(len(heat_val.index))); ax.set_yticklabels(heat_val.index)
ax.set_title("McNemar Significance Heatmap (-log10 p-value)")
fig.colorbar(im, ax=ax, label="-log10(p)")
fig.tight_layout()
fig.savefig(FIG_DIR / "mcnemar_significance_heatmap.png", dpi=180)
plt.close(fig)

for value_col, title, out_name in [
    ("false_negative_count", "False Negative Counts by Model and Split", "false_negative_counts_by_model_split.png"),
    ("false_positive_count", "False Positive Counts by Model and Split", "false_positive_counts_by_model_split.png"),
]:
    data = fn_breakdown if "negative" in value_col else fp_breakdown
    pivot = data.pivot(index="display_name", columns="split", values=value_col).reindex([DISPLAY_NAMES[m] for m in MODEL_ORDER])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(pivot.index)); width = 0.20
    for i, split in enumerate(SPLITS):
        ax.bar(x + (i - 1.5) * width, pivot[split], width, label=SPLIT_LABELS[split])
    ax.set_xticks(x); ax.set_xticklabels(pivot.index, rotation=35, ha="right")
    ax.set_title(title); ax.set_ylabel("Count"); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / out_name, dpi=180); plt.close(fig)

pivot_shift = adversarial_fn_shift.pivot(index="display_name", columns="adversarial_split", values="clean_tp_to_adversarial_fn_count").reindex([DISPLAY_NAMES[m] for m in MODEL_ORDER])
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pivot_shift.index)); width = 0.25
for i, split in enumerate(["test_adv_10", "test_adv_20", "test_adv_30"]):
    ax.bar(x + (i - 1) * width, pivot_shift[split], width, label=SPLIT_LABELS[split])
ax.set_xticks(x); ax.set_xticklabels(pivot_shift.index, rotation=35, ha="right")
ax.set_title("Clean TP to Adversarial FN Shift")
ax.set_ylabel("Count")
ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR / "adversarial_fn_shift_by_model.png", dpi=180); plt.close(fig)

if not weights_wide.empty:
    rank_wide = weights_wide[G_FEATURES].rank(axis=1, ascending=False)
    fig, ax = plt.subplots(figsize=(9, 3.5))
    im = ax.imshow(rank_wide.values, aspect="auto", cmap="magma_r")
    ax.set_xticks(range(len(G_FEATURES))); ax.set_xticklabels(G_FEATURES, rotation=45, ha="right")
    ax.set_yticks(range(len(rank_wide.index))); ax.set_yticklabels([f"seed {s}" for s in rank_wide.index])
    ax.set_title("GA Weight Rank Stability Across Seeds")
    fig.colorbar(im, ax=ax, label="Rank (1 = highest weight)")
    fig.tight_layout(); fig.savefig(FIG_DIR / "ga_weight_rank_stability_heatmap.png", dpi=180); plt.close(fig)

print("Validation figures saved:", FIG_DIR)

## Reports

In [ ]:
table_344 = mcnemar_df[[
    "split_label", "comparison", "statistic", "p_value", "method", "discordant_pairs",
    "significant_uncorrected_0_05", "significant_bonferroni_per_split", "significant_bonferroni_global"
]].copy()
(REPORT_DIR / "mcnemar_table_3_44.md").write_text(
    "# Table 3.44 Statistical Significance Table\n\n"
    "McNemar tests are computed on paired representative prediction rows using `final_row_id`. "
    "Bonferroni per-split alpha is 0.05/9. A stricter global 0.05/36 flag is also reported.\n\n"
    + table_344.to_markdown(index=False),
    encoding="utf-8",
)

error_report = f"""# Error Analysis Summary

This report summarizes representative-run confusion matrices, false negatives, false positives, and clean-to-adversarial false-negative shifts.

## Confusion Matrix Summary

```text
{confusion_df.to_string(index=False)}
```

## False Negative Counts

```text
{fn_breakdown.to_string(index=False)}
```

## False Positive Counts

```text
{fp_breakdown.to_string(index=False)}
```

## Clean TP to Adversarial FN Shift

```text
{adversarial_fn_shift.to_string(index=False)}
```
"""
(REPORT_DIR / "error_analysis_summary.md").write_text(error_report, encoding="utf-8")

ga_report = f"""# GA Interpretability Validation Summary

The thesis target for weight stability is Spearman rho >= 0.80 across seed runs.

## Spearman Rank Correlations

```text
{ga_corr.to_string(index=False)}
```

## Top Feature Group Stability

```text
{top_stability.to_string(index=False)}
```

## Selected Final Weights

```text
{selected_weights.to_string(index=False) if not selected_weights.empty else 'Unavailable'}
```

## Selected Threshold and Scale Metadata

```json
{json.dumps(selected_threshold, indent=2)}
```
"""
(REPORT_DIR / "ga_interpretability_validation_summary.md").write_text(ga_report, encoding="utf-8")

summary_report = f"""# Statistical Validation Summary

Notebook 06 is validation-only. It does not train models, load checkpoints, tune thresholds, or select models.

## Thesis Alignment

- Table 3.40: all nine pairwise comparisons are included.
- Table 3.44: McNemar statistic, p-value, discordant pairs, and Bonferroni significance are reported.
- Section 3.7.6: confusion matrices, false negatives, false positives, and adversarial false-negative shifts are summarized.
- Section 3.7.4: GA weight rank stability is summarized with Spearman correlations and top feature groups.

## Representative Run Rule

For stochastic models, the representative run is the median F1 seed on the same split. Ties use seed priority 42, then 7, then 123. Ablation C uses its only available run.

## Pairing Audit

All representative prediction files were paired by `final_row_id`; labels were checked for alignment before McNemar testing.

## Bonferroni Thresholds

- Per-split Bonferroni alpha: {ALPHA_PER_SPLIT_BONFERRONI:.8f}
- Global Bonferroni alpha: {ALPHA_GLOBAL_BONFERRONI:.8f}

## McNemar Summary

```text
{summary.to_string(index=False)}
```
"""
(REPORT_DIR / "statistical_validation_summary.md").write_text(summary_report, encoding="utf-8")
print("Reports saved:", REPORT_DIR)

## Create Statistical Validation ZIP

In [ ]:
CONTENT_OUT = Path("/content") if (os.name != "nt" and Path("/content").exists()) else PROJECT_ROOT
zip_output = CONTENT_OUT / "final_statistical_validation_outputs.zip"
if zip_output.exists():
    zip_output.unlink()
bundle_dir = CONTENT_OUT / "final_statistical_validation_outputs_bundle"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)
for rel in ["results/statistical_validation", "results/figures/statistical_validation", "reports/statistical_validation"]:
    src = PROJECT_ROOT / rel
    dst = bundle_dir / rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Adding:", src)
    else:
        print("Skipping missing folder:", src)
zip_path = shutil.make_archive(str(zip_output.with_suffix("")), "zip", root_dir=bundle_dir)
print("Final statistical validation ZIP created:")
print(zip_path)

## Final Checklist

In [ ]:
required = [
    OUT_DIR / "representative_seed_selection.csv",
    OUT_DIR / "prediction_pairing_audit.csv",
    OUT_DIR / "mcnemar_pairwise_results.csv",
    OUT_DIR / "mcnemar_contingency_tables.csv",
    OUT_DIR / "mcnemar_bonferroni_summary.csv",
    OUT_DIR / "confusion_matrix_representative_runs.csv",
    OUT_DIR / "false_negative_breakdown_by_model_split.csv",
    OUT_DIR / "false_positive_breakdown_by_model_split.csv",
    OUT_DIR / "adversarial_fn_shift_summary.csv",
    OUT_DIR / "ga_weight_spearman_rank_correlations.csv",
    OUT_DIR / "ga_weight_top_group_stability.csv",
    OUT_DIR / "statistical_validation_manifest.json",
    REPORT_DIR / "statistical_validation_summary.md",
    REPORT_DIR / "mcnemar_table_3_44.md",
    REPORT_DIR / "error_analysis_summary.md",
    REPORT_DIR / "ga_interpretability_validation_summary.md",
]
for p in required:
    print(f"{p.relative_to(PROJECT_ROOT)}: {'FOUND' if p.exists() else 'MISSING'}")
print("Training code present: NO")
print("Model checkpoint loading present: NO")
print("Threshold tuning present: NO")
print("ZIP:", zip_path)